## 函数式编程 Functional Programming

-   一个函数就可以接收另一个函数作为参数


In [ ]:
# 传入函数
def add(x, y, f):
    return f(x) + f(y)


print(add(5, -6, abs))

### 高阶函数 Higher-order function

-   变量可以指向函数


#### map(func, collection)

-   返回类型 map object


In [ ]:
# 传入函数依次作用到序列每个元素，并把结果作为新的Iterator返回
def upper_first_word(str):
    return str[:1].upper() + str[1:].lower()


print(list(map(upper_first_word, ['adam', 'LISA', 'barT'])))
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9]
print(list(map(str, numbers)))

print(numbers)

#### reduce


In [ ]:
# 第一次按照形参传入 把结果和序列下一个元素继续迭代方法
from functools import reduce


def add(x, y):
    return x + y


print(reduce(add, [1, 3, 5, 7, 9]))

In [ ]:
# 字符串作为字符数组做迭代
def str2int(str):
    # char->int
    def char2num(char):
        return {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}[char]

    return reduce(lambda x, y: x * 10 + y, map(char2num, str))


str2int('45666')

#### `filter(func, list)` 用 func 过滤 list 中每一个元素，True 保留，否则遗弃

-   传入函数依次作用于每个元素，根据返回值是 True 还是 False 决定保留还是丢弃该元素


In [ ]:
def not_empty(s):
    return s and s.strip()


print(list(filter(not_empty, ['A', '', 'B', None, 'C', '  '])))


# 求素数
def _odd_iter():
    n = 1
    while True:
        n = n + 2
        yield n


def _not_divisible(n):
    return lambda x: x % n > 0


def primes():
    yield 2
    it = _odd_iter()
    while True:
        n = next(it)
        yield n
        it = filter(_not_divisible(n), it)


for n in primes():
    if n < 50:
        print(n)
    else:
        break

In [ ]:
def is_palindrome(s):
    return str(s)[:] == str(s)[-1::-1]


output = filter(is_palindrome, range(1, 100))
print('1~1000:', list(output))
if list(filter(is_palindrome, range(1, 100))) == [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 22, 33, 44, 55, 66, 77, 88, 99]:
    print('测试成功!')
else:
    print('测试失败!')

#### sorted

-   内置 sorted()函数可以对 list 进行排序
-   key 指定函数将作用于 list 每一个元素上，根据 key 函数结果进行排序,按照原来值返回


In [ ]:
print(sorted([36, 5, -12, 9, -21], key=abs))

print(sorted(['bob', 'about', 'Zoo', 'Credit'], key=str.lower, reverse=True))

L = [('Bob', 75), ('Adam', 92), ('Bart', 66), ('Lisa', 88)]


def by_name(t):
    return t[0]


sorted(L, key=by_name)

### 函数作为结果值返回


In [ ]:
# 内部函数sum可以引用外部函数lazy_sum的参数和局部变量，当lazy_sum返回函数sum时，相关参数和变量都保存在返回的函数中
# 调用lazy_sum()时，每次调用都会返回一个新的函数，即使传入相同的参数
def lazy_sum(*args):
    def sum():
        ax = 0
        for n in args:
            ax = ax + n
        return ax

    return sum


f = lazy_sum(1, 3, 5, 7, 9)
f1 = lazy_sum(1, 3, 5, 7, 9)
print(f())
f == f1

### 闭包（Closure）

-   返回函数不要引用任何循环变量，或者后续会发生变化的变量
-   方法 再创建一个函数，用该函数的参数绑定循环变量当前值，无论该循环变量后续如何更改，已绑定到函数参数的值不变


In [ ]:
def count():
    def f(j):
        def g():
            return j * j

        return g

    fs = []
    for i in range(1, 4):
        fs.append(f(i))
    return fs


f1, f2, f3 = count()
f1(), f2(), f3()

### lambda 匿名函数


In [ ]:
# 冒号前面 x 表示函数参数,只能有一个表达式
# 可以把匿名函数赋值给一个变量，利用变量来调用该函数
def f(x): return x * x


print(f(5))


def build(x, y):
    return lambda: x * x + y * y


build(3, 4)()

In [ ]:
def func(x, y): return x * x + y * y


func(3, 4)

### 装饰器

-   修改原函数一些功能，使得原函数不需要修改


In [ ]:
# 方法名与参数分离
def func(message):
    def get_message(message):
        print('Got a message: {}'.format(message))

    return get_message(message)


func('hello world')

In [ ]:
# 调用封装
def func_closure():
    def get_message(message):
        print('Got a message: {}'.format(message))

    return get_message


send_message = func_closure()
send_message('hello world')

In [ ]:
# 原理
def my_decorator(func):
    def wrapper():
        print('wrapper of decorator')
        func()

    return wrapper


def greet():
    print('hello world')


greet = my_decorator(greet)
greet()

In [ ]:
# 上一种封装
def my_decorator(func):
    def wrapper():
        print('wrapper of decorator')
        func()

    return wrapper


@my_decorator
def greet():
    print('hello world')


greet()

In [ ]:
# 带有参数的装饰器
def my_decorator(func):
    def wrapper(message):
        print('wrapper of decorator')
        func(message)

    return wrapper


@my_decorator
def greet(message):
    print(message)


greet('hello world')

In [ ]:
# 带有自定义参数的装饰器
def repeat(num):
    def my_decorator(func):
        def wrapper(*args, **kwargs):
            for i in range(num):
                print('wrapper of decorator')
                func(*args, **kwargs)

        return wrapper

    return my_decorator


@repeat(4)
def greet(message):
    print(message)


greet(['hello world', 'Henry'])
print(greet.__name__)
help(greet)

In [ ]:
# 保留原函数的元信息
import functools


def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print('wrapper of decorator')
        func(*args, **kwargs)

    return wrapper


@my_decorator
def greet(message):
    print(message)


greet.__name__

In [ ]:
# 类装饰器 依赖于函数__call__()
class Count:
    def __init__(self, func):
        self.func = func
        self.num_calls = 0

    def __call__(self, *args, **kwargs):
        self.num_calls += 1
        print('num of calls is: {}'.format(self.num_calls))
        return self.func(*args, **kwargs)


@Count
def example():
    print("hello world")


example()

In [ ]:
# 嵌套
import functools


def my_decorator1(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print('execute decorator1')
        func(*args, **kwargs)

    return wrapper


def my_decorator2(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print('execute decorator2')
        func(*args, **kwargs)

    return wrapper


@my_decorator1
@my_decorator2
def greet(message):
    print(message)


greet('hello world')

#### 场景

-   有点像 AOP 应用场景 把一些常用的业务逻辑分离，提高程序可重用性，降低耦合度，提高开发效率
-   重复逻辑与业务逻辑剥离
-   使用纵向业务逻辑扩展
-   横向业务扩展：还是要分层


In [ ]:
# 身份认证:一级级验证
import functools


def authenticate(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        request = args[0]
        if check_user_logged_in(request):  # 如果用户处于登录状态
            return func(*args, **kwargs)  # 执行函数post_comment()
        else:
            raise Exception('Authentication failed')

    return wrapper


@authenticate
def post_comment(request, ...)


...


In [ ]:
# 日志记录
import time
import functools


def log_execution_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        res = func(*args, **kwargs)
        end = time.perf_counter()
        print('{} took {} ms'.format(func.__name__, (end - start) * 1000))
        return res

    return wrapper


@log_execution_time
def calculate_similarity(items):
    ...

In [ ]:
# 校验
import functools


def validation_check(input):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        ...  # 检查输入是否合法

# @validation_check
# def neural_network_training(param1, param2, ...):
#     ...

In [ ]:
# 缓存
@lru_cache
# def check(param1, param2, ...) # 检查用户设备类型，版本号等等
#     ...


### 偏函数

-   int()函数提供额外 base 参数，默认值 10。如果传入 base 参数，可以做 N 进制的转换
-   functools.partial 帮助创建一个偏函数
-   函数参数个数太多需要简化时，使用 functools.partial 可以创建一个新函数，固定住原函数部分参数，从而在调用时更简单


In [ ]:
import functools

int2 = functools.partial(int, base=2)
int2('1000000')